# Vegetation challenge: data exploration

Step 1: load the files and see what we have. Keep printed output to summaries (counts, column names). Do not print raw rows, because the data are not published.

In [18]:
from pathlib import Path

import pandas as pd

## Paths

The data live outside the repository. Change `data_dir` if the files move or you are someone reproducing this workflow from your own machine.

In [19]:
data_dir = Path("/Users/scohen/Documents/NaturalState/Take Home Project/Data Vegetation")

exports_dir = data_dir / "ODK Data Exports"
entities_dir = data_dir / "Entity lists"

print(exports_dir.exists(), entities_dir.exists())

True True


## Read the files

The ODK exports are the data to check. The entity lists are the reference data.

In [20]:
exports = {p.stem: pd.read_csv(p) for p in sorted(exports_dir.glob("*.csv"))}
entities = {p.stem: pd.read_csv(p) for p in sorted(entities_dir.glob("*.csv"))}

## Overview: rows and columns per file

In [21]:
def overview(tables):
    return pd.DataFrame(
        {"rows": {name: df.shape[0] for name, df in tables.items()},
         "columns": {name: df.shape[1] for name, df in tables.items()}}
    )

overview(exports)

,rows,columns
herbaceous_veg_survey-additional_species_repeat,1296,30
herbaceous_veg_survey-quadrat_repeat,640,21
herbaceous_veg_survey,32,52
register_vegetation_plots,31,82


In [22]:
overview(entities)

,rows,columns
areas,1,10
centroids,51,21
ecosystem_types,5,10
project_team,6,13
projects,1,11
species,370,21
species_extra,250,14
species_extra_ids,109,11
surveys,4,15
transport,5,10


### Load data tables

In [23]:
for name, df in {**exports, **entities}.items():
    globals()[name.replace("-", "_")] = df

### herbaceous_veg_survey

This is the main survey file. Each row is one survey submission: a field team's visit to one registered vegetation plot, recorded in ODK using the herbaceous vegetation form. The columns describe who did the survey (team and recorder), when, which plot and survey it belongs to, how many quadrats were recorded, and where the submission was made. The species found in each of the 20 quadrats are in the linked `quadrat_repeat` and `additional_species_repeat` files, which join back to this file through `KEY`. The data are used to assess herbaceous species richness and diversity.

In [24]:
form = pd.read_excel(data_dir / "ODK Data Collection Forms" / "herbaceous_veg_survey.xlsx", sheet_name="survey")
form = form.dropna(subset=["type"])

# Walk through the form, building each question's column name from its groups
groups, in_repeat, rows = [], 0, []
for _, q in form.iterrows():
    kind = str(q["type"]).strip()
    if kind == "begin_group":
        groups.append(q["name"])
    elif kind == "end_group":
        groups.pop()
    elif kind == "begin_repeat":
        in_repeat += 1
    elif kind == "end_repeat":
        in_repeat -= 1
    elif in_repeat == 0:          # questions inside repeats belong to the other files
        text = q["label"] if pd.notna(q["label"]) else "calculated: " + str(q["calculation"])
        rows.append({"column": "-".join(groups + [q["name"]]), "form_type": kind,
                     "description": str(text)[:100]})
form_cols = pd.DataFrame(rows)

# Location questions are exported as four columns
geo = form_cols[form_cols["form_type"].isin(["geopoint", "background-geopoint"])]
form_cols = pd.concat([form_cols] + [geo.assign(column=geo["column"] + "-" + s)
                      for s in ["Latitude", "Longitude", "Altitude", "Accuracy"]], ignore_index=True)

survey = herbaceous_veg_survey
schema = (pd.DataFrame({"column": survey.columns, "dtype": survey.dtypes.astype(str).values,
                        "missing": survey.isna().sum().values, "distinct": survey.nunique().values})
          .merge(form_cols, on="column", how="left"))
schema["form_type"] = schema["form_type"].fillna("ODK system column")
schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,SubmissionDate,str,0,32,ODK system column,NaN
1,note_overview,float64,32,0,note,This form is to be used by field data teams to...
2,field_team_specifics-selected_transport_type_uuid,str,0,1,select_one transport.csv,Select the the mode of transport:
3,field_team_specifics-transport_type_label,str,0,1,calculate,"calculated: instance(""transport"")/root/item[na..."
4,field_team_specifics-selected_project_team_uui...,str,0,2,select_multiple_from_file project_team.csv,Select all the team members involved in this c...
5,field_team_specifics-recorder_uuid,str,0,2,select_one_from_file project_team.csv,Who is recording the data:
6,field_team_specifics-recorder_label,str,0,2,calculate,"calculated: instance(""project_team"")/root/item..."
7,survey_begin-start_time,str,0,32,start,Start time
8,survey_begin-survey_year,int64,0,1,calculate,"calculated: format-date(${start_time}, '%Y')"
9,survey_begin-selected_survey_uuid,str,0,1,select_one_from_file surveys.csv,Select a Survey:


### Columns kept for the checks

`herbaceous_veg_survey` has 52 columns. I kept 22 of them: the ones needed to check who surveyed which plot, when, how many quadrats, where, and whether the submission is complete.

**What I kept, and the checks each group supports**

| Group | Why it is kept |
|---|---|
| Keys and links (`KEY`, plot columns) | Find repeated plot visits and confirm the plot is registered and viable |
| Team (recorder and team members) | Confirm the recorder is a valid team member |
| Timing (`SubmissionDate`, start and end time) | Check dates and survey duration |
| Survey context (survey, year, replicate) | Check the survey matches the plot's assigned survey |
| Sampling effort (quadrat count, quadrats with species) | Check for exactly 20 quadrats per plot |
| Location (latitude, longitude, accuracy) | Check the location is present, accurate, and near the plot |
| Completeness (attachments, review state, confirmation) | Check for missing photos and review flags |

**What I dropped**

- **Columns that never vary.** The project, area, survey type, target group, and transport labels each have a single value in all 32 rows, so they cannot show a problem.
- **Notes that are always empty.** The `note_*` and `quadrat_check_*` columns and the other note and heading fields are blank in every row.
- **ODK internals.** `base_active_herb_count`, `new_unknown_index_list`, `new_missing_index_list`, `meta-*`, `SubmitterID`, `Edits`, `FormVersion` and `Status` are system or calculation columns, not field observations. `DeviceID` is also dropped but could be kept to compare devices.

**Two things I noticed while choosing, to check later**

- There are 32 rows but only 30 distinct plots, so some plots were surveyed more than once.
- 3 submissions have no location (`background_geopoint` is empty).

In [25]:
keep = [
    "KEY", "plot_selection-selected_plot_uuid", "plot_selection-plot_name", "plot_selection-get_plot_status",
    "field_team_specifics-selected_project_team_uuid_multi", "field_team_specifics-recorder_uuid", "field_team_specifics-recorder_label",
    "SubmissionDate", "survey_begin-start_time", "survey_end-end_time",
    "survey_begin-selected_survey_uuid", "survey_begin-survey_year", "survey_begin-replicate",
    "observations-quadrat_repeat_count", "survey_end-quadrats_with_species",
    "survey_end-background_geopoint-Latitude", "survey_end-background_geopoint-Longitude", "survey_end-background_geopoint-Accuracy",
    "AttachmentsPresent", "AttachmentsExpected", "ReviewState", "survey_end-confirm_submission",
]

herbaceous_veg_survey_keep = herbaceous_veg_survey[keep]

### herbaceous_veg_survey-quadrat_repeat

This file holds the quadrat records. Each row is one quadrat (a 1 m × 1 m frame placed along the plot's transect) within one survey submission. Each survey should have 20 quadrats, so the 32 submissions give 640 rows. For each quadrat, the field team recorded whether any herbaceous species were present, which species from the approved list were selected, whether additional species were present, a photo, the GPS location, and an optional comment. It links to its survey through `PARENT_KEY` (matching the survey's `KEY`), and its species selections link to the `species` entity list. The species recorded here are the basis for assessing species richness and diversity.


In [27]:
def form_columns(form_path, repeat=None):
    form = pd.read_excel(form_path, sheet_name="survey").dropna(subset=["type"])
    stack, rows = [], []
    for _, q in form.iterrows():
        kind, name = str(q["type"]).strip(), q["name"]
        if kind == "begin_group":
            stack.append(("group", name)); continue
        if kind == "begin_repeat":
            stack.append(("repeat", name)); continue
        if kind in ("end_group", "end_repeat"):
            stack.pop(); continue
        repeats = [i for i, (t, n) in enumerate(stack) if t == "repeat"]
        innermost = stack[repeats[-1]][1] if repeats else None
        if innermost != repeat:
            continue
        start = repeats[-1] + 1 if repeats else 0
        groups = [n for t, n in stack[start:] if t == "group"]
        text = q["label"] if pd.notna(q["label"]) else "calculated: " + str(q["calculation"])
        rows.append({"column": "-".join(groups + [name]), "form_type": kind, "description": str(text)[:100]})
    cols = pd.DataFrame(rows)
    geo = cols[cols["form_type"].isin(["geopoint", "background-geopoint"])]
    return pd.concat([cols] + [geo.assign(column=geo["column"] + "-" + s)
                     for s in ["Latitude", "Longitude", "Altitude", "Accuracy"]], ignore_index=True)

def export_schema(df, form_cols):
    schema = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values,
                           "missing": df.isna().sum().values, "distinct": df.nunique().values})
    schema = schema.merge(form_cols, on="column", how="left")
    schema["form_type"] = schema["form_type"].fillna("ODK system column")
    return schema


survey_form = data_dir / "ODK Data Collection Forms" / "herbaceous_veg_survey.xlsx"

quadrat_schema = export_schema(
    herbaceous_veg_survey_quadrat_repeat,
    form_columns(survey_form, "quadrat_repeat"),
)
quadrat_schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,quadrat_number,int64,0,20,calculate,Quadrat Number
1,note_new_quadrat,float64,640,0,note,Move to **Quadrat ${quadrat_number}**
2,herbs_present,str,0,2,select_one yes_no,Are there any herbaceous species present?
3,herb_species-all_unknown_unidentifiable,str,7,2,select_one yes_no,Are **ALL** of the specimens either unidentifi...
4,herb_species-selected_herb_species_uuids,str,146,225,select_multiple species.csv,Select **ALL** of the species present:
5,herb_species-count_herb_species,float64,7,7,calculate,calculated: count-selected(${selected_herb_spe...
6,herb_species-note_herb_species_count,float64,640,0,note,**${count_herb_species}** species selected\n
7,additional_species_present,str,7,2,select_one yes_no,Are there any species present in the quadrat t...
8,species_list-known_species_list,str,146,188,calculate,calculated: if(${selected_herb_species_uuids} ...
9,species_list-additional_species_list,str,66,409,calculate,"calculated: join('<br/>', ${validated_name}[. ..."


### Columns kept for the checks: quadrat_repeat

`herbaceous_veg_survey-quadrat_repeat` has 21 columns. I kept 14: the ones needed to check that each survey has its 20 quadrats, that species and photos were recorded, and that locations are accurate.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `PARENT_KEY` | Identify each quadrat and link it to its survey |
| `quadrat_number` | Each survey should have quadrats 1 to 20, once each |
| `herbs_present`, `herb_species-all_unknown_unidentifiable` | Find quadrats with no species recorded, or where nothing could be matched to the species list |
| `herb_species-selected_herb_species_uuids`, `herb_species-count_herb_species` | The species recorded, and a cross-check that the count matches the number selected |
| `additional_species_present`, `species_list-all_species_list` | Link to the extra-species file, and a readable combined species list |
| `quadrat_image` | Check that a photo was taken for each quadrat |
| Latitude, longitude, accuracy | Check that the location is present, accurate (5 m or better per the SOP), and on the transect |
| `quadrat_comment` | Notes on unusual conditions (only a few are filled in) |

**What I dropped**

- **Empty notes.** `note_new_quadrat`, `herb_species-note_herb_species_count`, `species_list-note_species_summary_heading` and `species_list-note_species_list` are blank in all 640 rows.
- **Display-only lists.** `species_list-known_species_list` and `species_list-additional_species_list` are text the form builds from other columns, so they can be recreated from the species selections and the extra-species file.
- **Altitude.** The SOP sets no rule for altitude, so `location_quadrat-Altitude` supports no check.

In [28]:
quadrat_keep_cols = [
    "KEY", "PARENT_KEY", "quadrat_number",
    "herbs_present", "herb_species-all_unknown_unidentifiable",
    "herb_species-selected_herb_species_uuids", "herb_species-count_herb_species",
    "additional_species_present", "species_list-all_species_list",
    "quadrat_image",
    "location_quadrat-Latitude", "location_quadrat-Longitude", "location_quadrat-Accuracy",
    "quadrat_comment",
]

herbaceous_veg_survey_quadrat_repeat_keep = herbaceous_veg_survey_quadrat_repeat[quadrat_keep_cols]

herbaceous_veg_survey_quadrat_repeat_keep.shape   # should be (640, 14)

(640, 14)